In [41]:
import json
import os
import time
import requests

# Dataspace Authentication & Mutual Onboarding

This notebook performs the complete onboarding flow for the dataspace:
1. Configures agent URLs for Authority, Consumer, and Provider.
2. Links participant wallets.
3. Retrieves participant DIDs from `/.well-known/did.json`.
4. Requests credentials (`DataSpaceParticipant_jwt_vc_json`) from the Authority for both Consumer and Provider.
5. Performs mutual peer connection and token exchange between Consumer and Provider.
6. Retrieves and exports participant DIDs and access tokens for Dataspace Protocol communications.

## 1. Constants & Configuration
Configuration of constant variables used throughout the notebooks, including agent URLs, internal URLs, and participant IDs.

In [42]:
data_space_authority = os.environ.get("AUTHORITY_URL", "http://127.0.0.1:1500")
data_space_consumer = os.environ.get("CONSUMER_URL", "http://127.0.0.1:1100")
data_space_provider = os.environ.get("PROVIDER_URL", "http://127.0.0.1:1200")

internal_authority_url = os.environ.get("INTERNAL_AUTHORITY_URL", data_space_authority)
internal_consumer_url = os.environ.get("INTERNAL_CONSUMER_URL", data_space_consumer)
internal_provider_url = os.environ.get("INTERNAL_PROVIDER_URL", data_space_provider)

data_space_authority_participant_id = ""
data_space_provider_participant_id = ""
data_space_provider_participant_token = ""
data_space_provider_dsp_endpoint = ""
data_space_consumer_participant_id = ""
data_space_consumer_participant_token = ""
data_space_consumer_dsp_endpoint = ""

catalog_id = ""
dataset_id = ""
distribution_id = ""
data_service_id = ""
policy_id = ""
target_id = ""

cn_consumer_id = ""
cn_provider_id = ""
agreement_id = ""
transfer_consumer_id = ""
transfer_provider_id = ""
consumer_data_address = ""

json_header = {
    "Content-Type": "application/json",
}

print(f"Authority URL: {data_space_authority}")
print(f"Consumer URL:  {data_space_consumer}")
print(f"Provider URL:  {data_space_provider}")

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200


## 2. Link Wallets
Each participant links its wallet to initialize keys and DID documents.

In [43]:
for name, agent_url in [("Authority", data_space_authority), ("Consumer", data_space_consumer), ("Provider", data_space_provider)]:
    res = requests.post(f"{agent_url}/api/v1/wallet/link", headers=json_header, json={})
    if res.status_code in (200, 204):
        print(f"Wallet linked for {name} ({agent_url})")
    else:
        print(f"Warning linking wallet for {name}: {res.status_code} - {res.text}")

Wallet linked for Authority (http://127.0.0.1:1500)
Wallet linked for Consumer (http://127.0.0.1:1100)
Wallet linked for Provider (http://127.0.0.1:1200)


## 3. Retrieve Participant DIDs
Fetch the decentralized identifier (DID) for Authority, Consumer, and Provider from `/.well-known/did.json`.

In [44]:
data_space_authority_participant_id = requests.get(f"{data_space_authority}/.well-known/did.json").json()["id"]
data_space_consumer_participant_id = requests.get(f"{data_space_consumer}/.well-known/did.json").json()["id"]
data_space_provider_participant_id = requests.get(f"{data_space_provider}/.well-known/did.json").json()["id"]

print(f"Authority DID: {data_space_authority_participant_id}\n")
print(f"Consumer DID:  {data_space_consumer_participant_id}\n")
print(f"Provider DID:  {data_space_provider_participant_id}")

Authority DID: did:web:127.0.0.1%3A1500

Consumer DID:  did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ

Provider DID:  did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfR

## 4. Request Credentials from Authority
Consumer and Provider request verifiable credentials (`DataSpaceParticipant_jwt_vc_json`) from the Authority.

In [45]:
vc_type = "DataSpaceParticipant_jwt_vc_json"

def request_credential(agent_name, agent_url):
    res = requests.get(f"{agent_url}/api/v1/vc-request/all", headers=json_header)
    items = res.json().get("items", [])
    already_finalized = any(
        req.get("participant_id") == data_space_authority_participant_id
        and req.get("status") == "Finalized"
        and (vc_type in (req.get("vc_type_config") or []))
        for req in items
    )
    if already_finalized:
        print(f"{agent_name} already has {vc_type} credential from Authority.")
        return

    beg_body = {
        "url": f"{internal_authority_url}/api/v1/gate/access",
        "id": data_space_authority_participant_id,
        "nick": "authority",
        "vc_type": vc_type,
        "method": "cert",
        "auto": True,
    }
    beg_res = requests.post(f"{agent_url}/api/v1/vc-request/beg", json=beg_body, headers=json_header)
    if beg_res.status_code not in (200, 201, 204):
        raise RuntimeError(f"Error requesting credential for {agent_name}: {beg_res.status_code} - {beg_res.text}")

    res = requests.get(f"{agent_url}/api/v1/vc-request/all", headers=json_header)
    items = res.json().get("items", [])
    if not any(
        req.get("participant_id") == data_space_authority_participant_id
        and req.get("status") == "Finalized"
        and (vc_type in (req.get("vc_type_config") or []))
        for req in items
    ):
        raise RuntimeError(f"Credential issuance for {agent_name} did not finalize.")
    print(f"{agent_name} successfully obtained {vc_type} credential from Authority.")

request_credential("Consumer", data_space_consumer)
request_credential("Provider", data_space_provider)

Consumer already has DataSpaceParticipant_jwt_vc_json credential from Authority.
Provider already has DataSpaceParticipant_jwt_vc_json credential from Authority.


## 5. Mutual Peer Connection (Consumer <-> Provider)
Both participants establish mutual peer connections and exchange access tokens via their gate endpoints:
- Consumer connects to Provider (granting `data_space_provider_participant_token`)
- Provider connects to Consumer (granting `data_space_consumer_participant_token`)

In [46]:
def connect_peer(source_name, source_url, target_name, target_gate_url, target_did):
    res = requests.get(f"{source_url}/api/v1/peer-connection/request/all", headers=json_header)
    items = res.json().get("items", [])
    already_approved = any(
        req.get("participant_id") == target_did
        and req.get("status") == "Approved"
        and bool(req.get("final_token"))
        for req in items
    )
    if already_approved:
        print(f"{source_name} already connected and approved with {target_name}.")
        return

    connect_body = {
        "id": target_did,
        "nick": target_name,
        "url": target_gate_url,
        "actions": ["talk"],
        "visibility": "Public",
        "auto": True,
    }
    connect_res = requests.post(f"{source_url}/api/v1/peer-connection/connect", json=connect_body, headers=json_header)
    if connect_res.status_code not in (200, 201, 204):
        raise RuntimeError(f"Error initiating connection from {source_name} to {target_name}: {connect_res.status_code} - {connect_res.text}")

    print(f"Waiting for {source_name} to complete peer connection with {target_name}...")
    for _ in range(30):
        res = requests.get(f"{source_url}/api/v1/peer-connection/request/all", headers=json_header)
        matching = [req for req in res.json().get("items", []) if req.get("participant_id") == target_did]
        if matching:
            latest = matching[-1]
            status = latest.get("status")
            if status == "Approved" and latest.get("final_token"):
                print(f"{source_name} connected to {target_name} successfully.")
                return
            elif status == "Rejected":
                raise RuntimeError(f"Connection from {source_name} to {target_name} was rejected.")
        time.sleep(1)
    raise TimeoutError(f"Timeout waiting for {source_name} to authenticate with {target_name}")

# Consumer connects to Provider
connect_peer("Consumer", data_space_consumer, "provider", f"{internal_provider_url}/api/v1/gate/access", data_space_provider_participant_id)

# Provider connects to Consumer
connect_peer("Provider", data_space_provider, "consumer", f"{internal_consumer_url}/api/v1/gate/access", data_space_consumer_participant_id)

Consumer already connected and approved with provider.
Provider already connected and approved with consumer.


## 6. Access Tokens Retrieval
Retrieve the peer access tokens established during onboarding:
- `data_space_provider_participant_token`: Held by Consumer to present when calling Provider.
- `data_space_consumer_participant_token`: Held by Provider to present when calling Consumer.

In [47]:
# Extract Consumer -> Provider token from Consumer's requests
consumer_reqs = requests.get(f"{data_space_consumer}/api/v1/peer-connection/request/all", headers=json_header).json()
for req in consumer_reqs.get("items", []):
    if req.get("participant_id") == data_space_provider_participant_id and req.get("status") == "Approved":
        data_space_provider_participant_token = req.get("final_token", "")
        break

# Extract Provider -> Consumer token from Provider's requests
provider_reqs = requests.get(f"{data_space_provider}/api/v1/peer-connection/request/all", headers=json_header).json()
for req in provider_reqs.get("items", []):
    if req.get("participant_id") == data_space_consumer_participant_id and req.get("status") == "Approved":
        data_space_consumer_participant_token = req.get("final_token", "")
        break

print(f"Provider DID:                          {data_space_provider_participant_id}")
print(f"Provider token (held by Consumer):     {data_space_provider_participant_token}\n")
print(f"Consumer DID:                          {data_space_consumer_participant_id}")
print(f"Consumer token (held by Provider):     {data_space_consumer_participant_token}")

Provider DID:                          did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ
Provider token (held by Consumer):     6e3d_w2Ut9RXX2i95YCZgSR5YsQrzRypT8JRIzTG-dI

Consumer DID:                          did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXF